# AniNow Klima: analysis notebook

Computational companion to [docs/Paper.md](../docs/Paper.md). Section numbers follow the
paper. Every figure is drawn from files the model has already written, so nothing here
computes a score:

- `model/results.json` from `python model/klima_model.py`
- `docs/analysis/*.csv` from `python model/analysis.py --out docs/analysis`

Run both before this notebook.

This notebook and `make_figures.py` were drafted with AI assistance (Claude, Anthropic)
and reviewed by Team AniNow.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import pandas as pd
import make_figures as F

data = F.load()
seasons, places, crops, pairs, results = data
print(f"{len(places)} sites, {len(crops)} crops, {len(seasons):,} site-crop-season rows")
print(f"pass mark {results['meta']['pass_mark']}, window {results['meta']['window_days']} days")

## 2. Study sites and data

Twenty-two sites, each with its own NASA POWER record and a measured SARAI soil sample.
Sites in the same MERRA-2 grid cell share their weather.

In [ ]:
cols = ['place', 'latitude', 'longitude', 'weather_cell', 'soil_ph', 'soil_texture',
        'window_rain_median_mm', 'recommendation']
print(places['weather_cell'].nunique(), 'distinct weather cells')
places[cols]

In [ ]:
rated = pairs[pairs['status'] == 'rated']
scored = pd.to_numeric(seasons['suitability'], errors='coerce').notna().sum()
print(f"{len(rated)} rated site-crop pairs, {len(pairs) - len(rated)} excluded, "
      f"{scored:,} crop-seasons scored")

## 3. Methods

Three steps: score each condition against published limits, score water as supply over
demand, take the weakest and count the good years.

$$S = \min\bigl(\sigma_T,\ W,\ \sigma_{\mathrm{pH}},\ \tau\bigr), \qquad
W = \min\left(1,\ \frac{R}{ET_c}\right), \qquad
ET_c = \sum_{i} K_c(i)\,ET_0(i) + nP$$

$$ET_0 = 0.0023\,(T_{\mathrm{mean}} + 17.8)\,\sqrt{T_{\max} - T_{\min}}\;R_a$$

In [ ]:
fig, _ = F.fig_formulas(data); fig

In [ ]:
fig, _ = F.fig_how_it_decides(data); fig

In [ ]:
fig, _ = F.fig_the_function(data); fig

## 4. Results

### 4.1 Planting date at the pilot site

In [ ]:
fig, _ = F.fig_when_to_plant(data); fig

### 4.2 Crop comparison at the pilot site

In [ ]:
pilot = pairs[(pairs['place'] == F.PILOT) & (pairs['status'] == 'rated')]
show = pilot[pilot['crop'].isin(['mungbean', 'eggplant', 'tomato', 'rice'])]
show[['crop', 'growing_days', 'suitability', 'good_years', 'years_scored',
      'good_share_low95', 'good_share_high95', 'extra_water_mm', 'weakest_condition']]

In [ ]:
fig, _ = F.fig_seasons(data); fig

### 4.3 Across 22 sites

In [ ]:
fig, _ = F.fig_places(data); fig

### 4.4 Water saved against a second rice crop

In [ ]:
saved = pd.to_numeric(places['water_saved_against_rice_mm'], errors='coerce')
print(f"median {saved.median():.0f} mm, range {saved.min():.0f} to {saved.max():.0f} mm, "
      f"positive at {(saved > 0).sum()} of {len(saved)} sites")
fig, _ = F.fig_water_saved(data); fig

### 4.5 What limits suitability

In [ ]:
print(rated['weakest_condition'].value_counts().to_string())
fig, _ = F.fig_what_decides(data); fig

In [ ]:
fig, _ = F.fig_follows_rain(data); fig

In [ ]:
fig, _ = F.fig_margin(data); fig

### 4.6 Sensitivity to the pass mark and other chosen constants

In [ ]:
fig, _ = F.fig_pass_mark(data); fig

In [ ]:
fig, _ = F.fig_texture_cost(data); fig

### 4.7 How far the counts can be trusted

The record is not stationary, seasons are not independent, and the El Niño comparison is
under-powered. None of these is a performance figure: with no harvest record there is no
accuracy to report.

In [ ]:
fig, _ = F.fig_halves(data); fig

In [ ]:
fig, _ = F.fig_independence(data); fig

In [ ]:
fig, _ = F.fig_el_nino(data); fig

## Write every figure to `docs/figures/`

In [ ]:
out = os.path.join('..', 'docs', 'figures')
for fn in F.FIGURES:
    fig, name = fn(data)
    fig.savefig(os.path.join(out, name + '.png'), metadata=F.PNG_METADATA)
    F.plt.close(fig)
print(len(F.FIGURES), 'figures written')